# Medium Article Semantic Search by Title+Subtitle

### Load Data

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("data/medium_post_titles.csv") # excercise whole data set
# data source: https://www.kaggle.com/datasets/nulldata/medium-post-titles

In [5]:
df["subtitle_truncated_flag"].value_counts()

subtitle_truncated_flag
False    83029
True     43389
Name: count, dtype: int64

### Data Cleanup

In [6]:
df.head()

,category,title,subtitle,subtitle_truncated_flag
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False
3,equality,"""Call me Nat Love"" :Black Cowboys and the Fron...",NaN,False
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False


In [7]:
df.shape

(126418, 4)

In [8]:
# df.isna().sum()

df = df.dropna()
df = df[~df["subtitle_truncated_flag"]]
# df["subtitle_truncated_flag"].value_counts()

df['title_extended'] = df['title'] + df['subtitle']

In [9]:
# df.head()
# df['category'].nunique()  # metadata
# df.shape # 6k vectors, full set in excercise

### Prep for Upsert

In [10]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

True

In [11]:
# init pinecone
from pinecone import Pinecone, ServerlessSpec
# API_KEY = "YOUR API KEY"
pc = Pinecone(api_key=os.getenv("PINECONE"))

In [ ]:
pc.create_index(name = "medium-data", 
                dimension=384, 
                metric="cosine",
                spec=ServerlessSpec(
                    cloud="aws",
                    region="us-east-1"
                )) # remember to use only us-east-1 in free tier

In [12]:
pc.list_indexes()

[
    {
        "name": "medium-data",
        "metric": "cosine",
        "host": "medium-data-nm1v9fj.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    }
]

In [1]:
import torch
from sentence_transformers import SentenceTransformer

c:\Users\vasal\Study\VectorDB\.vecdb\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda') # cuda or cpu

In [13]:
df['values'] = df['title_extended'].map(
    lambda x: (model.encode(x)).tolist()) # python list, 6k rows 1 min

In [14]:
df.head()

,category,title,subtitle,subtitle_truncated_flag,title_extended,values
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False,"""21 Conversations"" - A fun (and easy) game for...","[-0.03107442893087864, -0.014303390868008137, ..."
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False,"""Biblical Porn"" at Mars HillAuthor and UW lect...","[-0.03467028960585594, -0.018165268003940582, ..."
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False,"""CISGENDER?! Is That A Disease?!""Or, a primer ...","[0.0374072901904583, -0.0008568316115997732, -..."
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we...","[-0.013686544261872768, 0.004296037834137678, ..."
5,cryptocurrency,"""Cypherpunks and Wall Street"": The Security To...",Bruce Fenton presents at the World Blockchain ...,False,"""Cypherpunks and Wall Street"": The Security To...","[-0.031468845903873444, -0.004646629095077515,..."


In [15]:
df['id'] = df.reset_index(drop = 'index').index

In [16]:
df['metadata'] = df.apply(lambda x: {
    'title' : x['title'],
    'subtitle': x['subtitle'],
    'category': x['category']
    
}, axis=1)

In [17]:
df_upsert = df[['id', 'values', 'metadata']]

In [18]:
df_upsert['id'] = df_upsert['id'].map(lambda x: str(x))

C:\Users\vasal\AppData\Local\Temp\ipykernel_8696\3006284790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_upsert['id'] = df_upsert['id'].map(lambda x: str(x))


In [19]:
index =pc.Index('medium-data')

In [20]:
index.upsert_from_dataframe(df_upsert) # 6k takes 1 min

sending upsert requests: 100%|██████████| 81545/81545 [03:15<00:00, 417.18it/s]


UpsertResponse(upserted_count=81545, _response_info={'raw_headers': {'date': 'Sat, 27 Dec 2025 20:45:17 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '164', 'x-pinecone-request-logical-size': '76250', 'x-pinecone-request-latency-ms': '197', 'x-pinecone-request-id': '6012218652815591617', 'x-envoy-upstream-service-time': '194', 'x-pinecone-response-duration-ms': '203', 'grpc-status': '0', 'server': 'envoy'}})

### Query

In [21]:
xc = index.query(vector=(model.encode("which city is the most beautiful")).tolist(), # python list
           top_k=10,
           include_metadata=True) 

In [22]:
for result in xc['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['title']}: {result['metadata']['category']} ")

0.57: 3 Places Where You Can Find Beauty: photography 
0.57: The 5 Best Cities For Street Art: art 
0.54: Morocco. It’s Pretty. It’s Also Pretty Ugly.: travel 
0.54: For The Simple Beauties Of Life — Photos: photography 
0.53: Delhi, The Murderous City.: travel 
0.53: The Best And Worst City To Rent: money 
0.51: The Poetry and History of Cities: cities 
0.5: The ‘Best Cities For Singles’ Are Wherever They Live Right Now: relationships 
0.5: The first city in the world that is switching to Bitcoin: economy 
0.48: Porto — where the hordes are smaller and the delights are huge: travel 


In [23]:
for result in xc['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['subtitle']}: {result['metadata']['category']} ")

0.57: If you are willing to look hard enough, eventually you will see beauty in the most difficult of places.: photography 
0.57: Incredible murals around the globe: art 
0.54: Highlights of my recent trip to the desert and the souks.: travel 
0.54: Autumn In The North, All The More Beautiful For Its Brevity: photography 
0.53: I Love This Place, But It Is Killing Us All: travel 
0.53: How Does Your City Rank?: money 
0.51: Far from being a soulless commercial center, the city is the most intense expression of humanity around.: cities 
0.5: So stop telling one-third of the population to move or risk dying alone: relationships 
0.5: You’ll never guess the world’s most expensive city.: economy 
0.48: The pretty pint-sized city is cheaper and quieter than Lisbon  but — with delicious food, wine and culture aplenty — is just as magical.: travel 


### Excercise: Upsert all data

In [25]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = model.encode(sentences)

In [27]:
len(embeddings[1])

384